# MP1 · Prompt Lab — Compare LLM Strategies on a Task
## Setup

In [1]:
pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#import and check
import openai
print("OpenAI ersion:",openai.__version__)
from openai import AsyncOpenAI
print("AsyncOpenAI imported successfully")

OpenAI ersion: 2.31.0
AsyncOpenAI imported successfully


In [3]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import asyncio
import json
import os
import time
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI

# Load environment variables from project-level .env file
env_path = Path.cwd() / ".env"

if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

load_dotenv(env_path, override=True)

print(f"Loaded environment variables from: {env_path}")

# Read API configuration
API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL")

assert API_KEY, "OPENAI_API_KEY is missing from .env"
assert BASE_URL, "OPENAI_BASE_URL is missing from .env"

# Initialize Vocareum-compatible OpenAI client
client = AsyncOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

# Model configuration
MODEL = os.getenv("MODEL", "gpt-4o-mini")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gpt-4o")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Cost rates ($ per token)
RATES = {
    "gpt-4o-mini": {
        "in": 0.15 / 1_000_000,
        "out": 0.60 / 1_000_000
    },
    "gpt-4o": {
        "in": 2.50 / 1_000_000,
        "out": 10.00 / 1_000_000
    }
}

print("API key loaded:", bool(API_KEY))
print("Base URL:", BASE_URL)
print("Extraction model:", MODEL)
print("Judge model:", JUDGE_MODEL)
print("Temperature:", TEMPERATURE)
print("Setup complete.")


Loaded environment variables from: /voc/work/Week 5_Graded Mini Project_HaimantiNag/MP1_PromptLab/mp1_prompt_lab/.env
API key loaded: True
Base URL: https://openai.vocareum.com/v1
Extraction model: gpt-4o-mini
Judge model: gpt-4o
Temperature: 0.0
Setup complete.


## Step 1 — Load the data

In [5]:
from pathlib import Path
import json

# Set data directory
DATA_DIR = Path("../data")

# Load job snippets
snippets = [
    json.loads(line)
    for line in (DATA_DIR / "job_snippets.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# Load golden dataset
golden = {
    row["id"]: row
    for row in (
        json.loads(line)
        for line in (DATA_DIR / "golden_set.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
}

# Display loading results
print(f"Loaded {len(snippets)} snippets, {len(golden)} golden entries.")

if snippets:
    print("Sample snippet:", snippets[0])

if golden:
    print("Sample golden entry:", next(iter(golden.values())))


Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}
Sample golden entry: {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean — clear company + role + years'}


## Step 2 — Write the four prompt strategies
Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

In [6]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the following information from the job description.

Return a valid JSON object with exactly these fields:
- company: Company name
- role: Job title
- years_experience_required: Minimum required years of experience (integer or null)

Rules:
- Extract only information explicitly supported by the text.
- Convert written numbers into integers.
- For experience ranges, use the minimum value.
- For "X+" or "at least X", use X.
- If the description explicitly states that no prior experience is required, use 0.
- If no specific experience requirement is mentioned, return null.
- Do not include explanations or additional text.

Job description:
{snippet_text}
"""
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include three worked examples."""

    return [
        {
            "role": "user",
            "content": f"""
Extract company, role, and years_experience_required
from job descriptions.

Follow these examples:

Example 1:
Input: "Acme Corp is hiring a Senior Software Engineer.
The ideal candidate must have 5+ years of experience."

Output:
{{"company": "Acme Corp", "role": "Senior Software Engineer",
"years_experience_required": 5}}

Example 2:
Input: "Northwind Ltd is hiring a Data Analyst.
Candidates must have at least two years of experience."

Output:
{{"company": "Northwind Ltd", "role": "Data Analyst",
"years_experience_required": 2}}

Example 3:
Input: "Cyberdyne Systems is hiring an AI/ML Research Scientist.
No previous work experience is required."

Output:
{{"company": "Cyberdyne Systems", "role": "AI/ML Research Scientist",
"years_experience_required": 0}}

Now extract the information from this job description.

Return only a valid JSON object with exactly these fields:
company, role, years_experience_required.

Rules:
- Use the minimum value for experience ranges.
- Convert written numbers into integers.
- Return null if no specific experience requirement is stated.
- Never invent information.

Job description:
{snippet_text}
"""
        }
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based."""

    return [
        {
            "role": "system",
            "content": """
You are an expert recruitment data extraction specialist.

Your task is to extract structured information from job descriptions.

Follow these instructions:
1. Identify the company name.
2. Identify the job title.
3. Identify the minimum required years of experience.
4. Convert written numbers into integers.
5. For experience ranges, select the minimum value.
6. Treat "X+" and "at least X" as X.
7. If the description explicitly states that no prior experience is required, return 0.
8. If no specific years requirement is stated, return null.
9. Do not treat preferred or desirable experience as a mandatory requirement unless the text explicitly makes it required.
10. Never invent missing information.

Output must be valid JSON with exactly these fields:
{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}

Do not include Markdown, explanations, or additional fields.
"""
        },
        {
            "role": "user",
            "content": f"""
Extract the information from this job description:

{snippet_text}
"""
        }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought."""

    return [
        {
            "role": "user",
            "content": f"""
Analyze the job description carefully before producing your answer.

Consider these steps internally:
1. Identify the company name.
2. Identify the job title.
3. Find any explicit years of experience requirement.
4. Interpret phrases such as "5+ years", "at least 4 years",
   and written-out numbers.
5. If a range is given, select its minimum value.
6. If the description explicitly states that no prior experience
   is required, use 0.
7. If no specific experience requirement exists, use null.
8. Verify that the extracted values are supported by the text.

Return ONLY a valid JSON object with exactly these fields:

{{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}}

Do not include your reasoning or any additional text.

Job description:
{snippet_text}
"""
        }
    ]


# Register all four prompt strategies
STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

print("Step 2: All four prompt strategies defined successfully.")


Step 2: All four prompt strategies defined successfully.


## Step 3 — Async batching
Run all 10 snippets × 4 strategies = 40 calls in parallel.
Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

In [7]:

# STEP 3 — ASYNC BATCHING

import asyncio
import json
import time


# 1. Parse the model response into a Python dictionary
def parse_response(text: str) -> dict | None:
    """Convert the model's response into a Python dictionary."""

    if not text:
        return None

    cleaned = text.strip()

    # Remove Markdown code fences
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        if lines:
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    # Find the beginning of the JSON object
    start = cleaned.find("{")

    if start == -1:
        return None

    try:
        result, _ = json.JSONDecoder().raw_decode(cleaned[start:])

        if isinstance(result, dict):
            return result

        return None

    except json.JSONDecodeError:
        return None


# 2. Limit simultaneous API requests
# All 40 tasks are created, but at most 5 requests run at once.
semaphore = asyncio.Semaphore(5)


# 3. Run one strategy on one job snippet
async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Execute one strategy-snippet combination and capture results."""

    snippet_id = snippet.get("id")
    snippet_text = snippet.get("snippet", "")

    start_time = time.perf_counter()

    try:
        # Generate prompt messages
        messages = STRATEGIES[strategy_name](snippet_text)

        # Make asynchronous API call
        async with semaphore:
            response = await client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=TEMPERATURE
            )

        # Calculate latency
        latency_seconds = time.perf_counter() - start_time

        # Extract raw response
        raw_response = response.choices[0].message.content or ""

        # Parse response
        parsed = parse_response(raw_response)

        # Extract token usage
        usage = response.usage

        prompt_tokens = usage.prompt_tokens if usage else 0
        completion_tokens = usage.completion_tokens if usage else 0

        # Calculate cost when usage and model rates are available
        model_rates = RATES.get(MODEL)

        if usage and model_rates:
            cost_usd = (
                prompt_tokens * model_rates["in"]
                + completion_tokens * model_rates["out"]
            )
        else:
            cost_usd = None

        return {
            "strategy_name": strategy_name,
            "snippet_id": snippet_id,
            "snippet": snippet_text,
            "raw_response": raw_response,
            "parsed_extraction": parsed,
            "cost_usd": cost_usd,
            "latency_sec": latency_seconds
        }

    except Exception as e:
        # Capture errors without stopping the remaining tasks
        latency_seconds = time.perf_counter() - start_time

        return {
            "strategy_name": strategy_name,
            "snippet_id": snippet_id,
            "snippet": snippet_text,
            "raw_response": f"ERROR: {str(e)}",
            "parsed_extraction": None,
            "cost_usd": None,
            "latency_sec": latency_seconds
        }


# 4. Run all four strategies across all snippets
async def run_all() -> list[dict]:
    """Run all strategy-snippet combinations asynchronously."""

    tasks = []

    for strategy_name in STRATEGIES:
        for snippet in snippets:
            tasks.append(
                run_one(strategy_name, snippet)
            )

    print(f"Starting {len(tasks)} API calls...")

    results = await asyncio.gather(*tasks)

    successful = sum(
        1 for result in results
        if result["parsed_extraction"] is not None
    )

    failed = len(results) - successful

    print(f"Completed {len(results)} calls.")
    print(f"Successfully parsed: {successful}")
    print(f"Failed or unparsed: {failed}")

    return results

print("Step 3 functions defined successfully.")


Step 3 functions defined successfully.


In [8]:
# Run it
results = await run_all()
print("\nTotal results:", len(results))
if results:
    print("\nSample result:")
    print(json.dumps(results[0], indent=2))

Starting 40 API calls...
Completed 40 calls.
Successfully parsed: 40
Failed or unparsed: 0

Total results: 40

Sample result:
{
  "strategy_name": "zero_shot",
  "snippet_id": "j01",
  "snippet": "Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.",
  "raw_response": "```json\n{\n  \"company\": \"Acme Corp\",\n  \"role\": \"Senior Software Engineer\",\n  \"years_experience_required\": 5\n}\n```",
  "parsed_extraction": {
    "company": "Acme Corp",
    "role": "Senior Software Engineer",
    "years_experience_required": 5
  },
  "cost_usd": 4.6649999999999996e-05,
  "latency_sec": 2.292259535000994
}


## Step 4 — Score against the golden set
Three scores per (strategy × snippet) pair:
1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

In [9]:
gold_lookup = golden
snippet_lookup = {
    item["id"]: item
    for item in snippets
}

In [10]:
async def score_llm_judge(extracted, gold):

    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise ValueError("OPENAI_API_KEY not found")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


In [11]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Return the number of correctly extracted fields (0–3)."""

    if extracted is None:
        return 0

    fields = [
        "company",
        "role",
        "years_experience_required"
    ]

    score = 0

    for field in fields:
        extracted_value = extracted.get(field)
        gold_value = gold.get(field)

        # Normalize string values before comparison
        if isinstance(extracted_value, str):
            extracted_value = extracted_value.strip().casefold()

        if isinstance(gold_value, str):
            gold_value = gold_value.strip().casefold()

        if extracted_value == gold_value:
            score += 1

    return score

In [12]:
def build_judge_prompt(snippet, extracted, gold):
    """Create the LLM judge evaluation prompt."""
    return f"""
You are an expert evaluator of job description information extraction.

Evaluate the extracted information against the reference answer.

JOB SNIPPET:
{snippet}

EXTRACTED ANSWER:
{json.dumps(extracted, ensure_ascii=False)}

REFERENCE ANSWER:
{json.dumps(gold, ensure_ascii=False)}

Evaluate these three fields:
1. company
2. role
3. years_experience_required

Use this scoring rubric:
1 = Mostly incorrect; little useful information is correct.
2 = Partially correct; some fields are correct, but important errors exist.
3 = Mostly correct; minor errors or omissions exist.
4 = Fully correct; all three fields match the reference.

Return ONLY a single integer from 1 to 4.
"""


judge_semaphore = asyncio.Semaphore(5)


async def score_llm_judge(snippet, extracted, gold):
    """Ask gpt-4o to score one extraction from 1 to 4."""

    if not isinstance(extracted, dict):
        return 1

    prompt = build_judge_prompt(snippet, extracted, gold)

    try:
        async with judge_semaphore:
            response = await client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=0
            )

        judge_text = (
            response.choices[0].message.content or ""
        ).strip()

        match = re.search(r"\b([1-4])\b", judge_text)

        if match:
            return int(match.group(1))

        return None

    except Exception as e:
        print(f"Judge evaluation error: {e}")
        return None


async def score_one_result(result):
    """Calculate all three scores for one strategy × snippet pair."""

    snippet_id = result["snippet_id"]
    extracted = result["parsed_extraction"]

    gold = gold_lookup.get(snippet_id)

    if gold is None:
        raise KeyError(
            f"No golden-set entry found for snippet ID: {snippet_id}"
        )

    parse_success = isinstance(extracted, dict)

    accuracy = score_accuracy(extracted, gold)

    llm_judge_score = await score_llm_judge(
        snippet=result["snippet"],
        extracted=extracted,
        gold=gold
    )

    scored_result = result.copy()

    scored_result["accuracy"] = accuracy
    scored_result["parse_success"] = parse_success
    scored_result["llm_judge_score"] = llm_judge_score

    return scored_result


async def apply_scoring(results):
    """Score all strategy × snippet results."""

    print(f"Scoring {len(results)} results...")

    scored = await asyncio.gather(
        *[score_one_result(result) for result in results]
    )

    print("Scoring completed.")

    return scored


# Run Step 4
scored = await apply_scoring(results)

print("\nFirst scored result:")
print(json.dumps(scored[0], indent=2))


Scoring 40 results...
Scoring completed.

First scored result:
{
  "strategy_name": "zero_shot",
  "snippet_id": "j01",
  "snippet": "Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.",
  "raw_response": "```json\n{\n  \"company\": \"Acme Corp\",\n  \"role\": \"Senior Software Engineer\",\n  \"years_experience_required\": 5\n}\n```",
  "parsed_extraction": {
    "company": "Acme Corp",
    "role": "Senior Software Engineer",
    "years_experience_required": 5
  },
  "cost_usd": 4.6649999999999996e-05,
  "latency_sec": 2.292259535000994,
  "accuracy": 3,
  "parse_success": true,
  "llm_judge_score": 4
}


In [13]:
# Quick Validation

import pandas as pd

df = pd.DataFrame(scored)

print("Total scored records:", len(df))
print("\nAccuracy distribution:")
print(df["accuracy"].value_counts().sort_index())

print("\nParse success rate:")
print(df["parse_success"].mean())

print("\nJudge score distribution:")
print(df["llm_judge_score"].value_counts().sort_index())

Total scored records: 40

Accuracy distribution:
accuracy
2     3
3    37
Name: count, dtype: int64

Parse success rate:
1.0

Judge score distribution:
llm_judge_score
4    40
Name: count, dtype: int64


## Step 5 — Build the comparison table

In [14]:
# Convert scored results into a DataFrame
df = pd.DataFrame(scored)

# Convert judge score from 1–4 scale to 1–25 scale
# 1 becomes 1, 2 becomes 9, 3 becomes 17, and 4 becomes 25.
df["judge_score_25"] = df["llm_judge_score"].apply(
    lambda score: 1 + (score - 1) * 8 if pd.notna(score) else None
)

# Aggregate metrics by strategy
comparison = df.groupby("strategy_name").agg(
    accuracy_mean=("accuracy", "mean"),
    parse_success_rate=("parse_success", "mean"),
    judge_score_mean=("judge_score_25", "mean"),
    total_cost_usd=("cost_usd", "sum"),
    median_latency_sec=("latency_sec", "median")
).reset_index()

# Format strategy names for display
strategy_names = {
    "zero_shot": "Zero shot",
    "few_shot": "Few shot",
    "structured": "Structured / Role based",
    "cot": "COT"
}

comparison["strategy_name"] = comparison["strategy_name"].map(
    lambda name: strategy_names.get(name, name)
)

# Format values for readability
comparison["accuracy_mean"] = comparison["accuracy_mean"].round(2)
comparison["parse_success_rate"] = (
    comparison["parse_success_rate"] * 100
).round(2)

comparison["judge_score_mean"] = comparison["judge_score_mean"].round(2)
comparison["total_cost_usd"] = comparison["total_cost_usd"].round(6)
comparison["median_latency_sec"] = comparison["median_latency_sec"].round(3)

# Rename columns for the final comparison table
comparison = comparison.rename(columns={
    "strategy_name": "Strategy",
    "accuracy_mean": "Mean Accuracy (0–3)",
    "parse_success_rate": "Parse Success (%)",
    "judge_score_mean": "Mean Judge Score (1–25)",
    "total_cost_usd": "Total Cost (USD)",
    "median_latency_sec": "Median Latency (seconds)"
})

# Display final comparison table
display(comparison)

,Strategy,Mean Accuracy (0–3),Parse Success (%),Mean Judge Score (1–25),Total Cost (USD),Median Latency (seconds)
0,COT,2.9,100.0,25.0,0.000520,8.903
1,Few shot,3.0,100.0,25.0,0.000602,4.423
2,Structured / Role based,2.9,100.0,25.0,0.000565,6.486
3,Zero shot,2.9,100.0,25.0,0.000464,2.358


## Step 6 — Write your reflection
Reflection written in `mp1_writeup.md` . Also the four questions from the brief are answerred there.